# Phase 4: Feature Engineering & Preprocessing Pipeline Verification

**Project Title**: AI-Based Sales Conversion Probability Prediction  
**Dataset**: UCI Online Shoppers Purchasing Intention Dataset  
**Module Verification**: `src/data_preprocessing.py` & `src/feature_engineering.py`  

---  
## Purpose
This notebook demonstrates and validates the configuration-driven preprocessing and feature engineering pipelines.  
It verifies:
1. Immutable raw data loading & SHA-256 hash preservation.
2. Pre-split deduplication (dropping 125 exact duplicate rows prior to splitting).
3. Stratified train/test splitting (`X_train`, `X_test`, `y_train`, `y_test`).
4. Configuration-driven feature alternatives (Config A, B, C, D).
5. Strict `PageValues` sensitivity toggle (`include_pagevalues=True` vs `False`).
6. Decoupled Logistic Regression vs. Random Forest preprocessing pipelines.
7. Zero data leakage (fitting strictly on `X_train`).

In [ ]:
import sys
import os
import hashlib
import numpy as np
import pandas as pd

sys.path.append("..")
from src.data_preprocessing import load_raw_data, handle_duplicates, split_data, build_preprocessing_pipeline
from src.feature_engineering import FeatureEngineeringTransformer

# 1. Raw Data Immutability Check
raw_path = "../data/raw/online_shoppers_intention.csv"
df_raw = load_raw_data(raw_path)
with open(raw_path, "rb") as f:
    sha256_hash = hashlib.sha256(f.read()).hexdigest()

print(f"Raw Dataset Shape: {df_raw.shape}")
print(f"SHA-256 Hash: {sha256_hash}")
assert sha256_hash == "b3055ee355f59134d851d32641183cb4a8b45def7124d2f50442a042f358e0d9", "Raw dataset hash mismatch!"

## 2. Pre-Split Deduplication & Stratified Train/Test Split

In [ ]:
# Pre-split deduplication
df_dedup, num_dups = handle_duplicates(df_raw)
print(f"Duplicates Dropped: {num_dups} (Remaining unique rows: {len(df_dedup)})")
assert num_dups == 125, f"Expected 125 duplicates, found {num_dups}"

# Stratified Split
X_train, X_test, y_train, y_test = split_data(df_dedup, save_processed=True, processed_dir="../data/processed")
print(f"X_train Shape: {X_train.shape}")
print(f"X_test Shape: {X_test.shape}")
print(f"y_train Base Rate: {y_train.mean()*100:.2f}%")
print(f"y_test Base Rate: {y_test.mean()*100:.2f}%")
assert "Revenue" not in X_train.columns, "Target column leaked into X_train!"
assert "Revenue" not in X_test.columns, "Target column leaked into X_test!"

## 3. Verification of Feature Configurations (Config A, B, C, D)

In [ ]:
# Test FeatureEngineeringTransformer on X_train
fe_config_a = FeatureEngineeringTransformer(page_count_config="original", duration_config="original")
df_a = fe_config_a.fit_transform(X_train)
print("Config A Columns:", list(df_a.columns[:8]))
assert "Total_Pages" not in df_a.columns
assert "Total_Duration" not in df_a.columns

fe_config_d = FeatureEngineeringTransformer(page_count_config="total", duration_config="total")
df_d = fe_config_d.fit_transform(X_train)
print("Config D Columns:", [c for c in df_d.columns if "Total" in c])
assert "Total_Pages" in df_d.columns
assert "Total_Duration" in df_d.columns
assert "Administrative" not in df_d.columns, "Component page count coexisted with Total_Pages!"
assert "Administrative_Duration" not in df_d.columns, "Component duration coexisted with Total_Duration!"

## 4. Verification of `PageValues` Sensitivity Toggle

In [ ]:
# Model Set A: include_pagevalues=True
fe_set_a = FeatureEngineeringTransformer(include_pagevalues=True)
df_set_a = fe_set_a.fit_transform(X_train)
assert "PageValues" in df_set_a.columns, "PageValues missing in Model Set A!"

# Model Set B: include_pagevalues=False
fe_set_b = FeatureEngineeringTransformer(include_pagevalues=False)
df_set_b = fe_set_b.fit_transform(X_train)
assert "PageValues" not in df_set_b.columns, "PageValues leaked into Model Set B!"
assert not any("PageValues" in col for col in df_set_b.columns), "Derived PageValues feature leaked into Model Set B!"
print("PageValues sensitivity toggle successfully verified!")

## 5. Preprocessing Pipelines Verification (Logistic Regression vs. Random Forest)

In [ ]:
# Build LR Pipeline
lr_pipe = build_preprocessing_pipeline(model_type="logistic_regression", include_pagevalues=True)
X_train_lr = lr_pipe.fit_transform(X_train)
print(f"LR Pipeline Processed Matrix Shape: {X_train_lr.shape}")

# Build RF Pipeline
rf_pipe = build_preprocessing_pipeline(model_type="random_forest", include_pagevalues=True)
X_train_rf = rf_pipe.fit_transform(X_train)
print(f"RF Pipeline Processed Matrix Shape: {X_train_rf.shape}")

# Verify test set transformation without fitting
X_test_lr = lr_pipe.transform(X_test)
X_test_rf = rf_pipe.transform(X_test)
print(f"X_test LR Processed Shape: {X_test_lr.shape}")
print(f"X_test RF Processed Shape: {X_test_rf.shape}")
print("\n=== ALL PHASE 4 PIPELINE VERIFICATIONS PASSED CLEANLY! ===")